# GaiaLab Naija Assistant — QLoRA notebook

This notebook prepares an explicitly licensed dataset and, only when you run the final training cell, fine-tunes a LoRA adapter. The 20 bundled rows are schema demonstrations, not a production corpus. **No training results are claimed.** Use a Python 3.11 GPU runtime and review all data and model licences first.

## 1. Runtime and repository setup

In Colab choose Runtime → Change runtime type → GPU. In Kaggle enable both a GPU and Internet in notebook settings. The setup cell below clones the public repository when the notebook is not already running from its root.

In [ ]:
import platform, torch
print('Python:', platform.python_version())
if platform.python_version_tuple()[:2] != ('3', '11'):
    print('WARNING: this project targets Python 3.11; select a 3.11 runtime for a supported run.')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, subprocess
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / 'requirements.txt').is_file():
    repo_root = Path.cwd() / 'gaialab-naija-assistant'
    if not repo_root.is_dir():
        subprocess.run([
            'git', 'clone',
            'https://github.com/oluwafemidiakhoa/gaialab-naija-assistant.git',
            str(repo_root),
        ], check=True)
os.chdir(repo_root)
assert Path('requirements.txt').is_file(), 'Repository setup failed.'
print('Repository root:', repo_root.resolve())
%pip install -q -r requirements.txt

## 2. Validate provenance and create splits

The validator fails on missing, unexpected, empty, placeholder, or duplicate-key fields. Dataset preparation also rejects exact records shared across train and validation. Read the report and inspect the actual rows before continuing.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'src.validate_dataset', 'data/sample_training_data.jsonl', '--output-dir', 'prepared_data'], check=True)
subprocess.run([sys.executable, '-m', 'src.prepare_dataset', '--train', 'prepared_data/train.jsonl', '--validation', 'prepared_data/validation.jsonl', '--output-dir', 'prepared_data/hf', '--overwrite'], check=True)

In [ ]:
import json
from pathlib import Path

rows = [json.loads(line) for line in Path('data/sample_training_data.jsonl').read_text(encoding='utf-8').splitlines()]
assert len(rows) == 20, 'The bundled demonstration set should contain exactly 20 rows.'
print('Review these sources/licences before training:', sorted({(r['source'], r['license']) for r in rows})[:3], '...')
print('Languages:', sorted({r['language'] for r in rows}))
print('Categories:', sorted({r['category'] for r in rows}))

## 3. Configure an explicit QLoRA run

Replace the demonstrations with a sufficiently large, consented, reviewed, and licensed dataset before meaningful training. The default model is small enough for many free GPU sessions, but availability and memory vary. Running the next cell starts training and may take time.

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
OUTPUT_DIR = 'outputs/gaialab-naija-adapter'
print('Model:', MODEL_ID)
print('Adapter output:', OUTPUT_DIR)

In [ ]:
# EXPLICIT TRAINING CELL — do not run until data and licences have been reviewed.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'src.train_qlora', '--model-id', MODEL_ID, '--dataset-dir', 'prepared_data/hf', '--output-dir', OUTPUT_DIR, '--epochs', '1'], check=True)

## 4. Review and optional publishing

Download the adapter first. Evaluate it with `python -m evaluation.evaluate_model`, document failures, and create a complete model card. If publishing, store `HF_TOKEN` in Colab/Kaggle secrets, start with a private Hugging Face repository, and push only after checking base-model and dataset licences. Never place a token in this notebook.